# Árvores e ensembles — quando a reta não dá conta

**Capítulo II.5** do livro vivo [Ciência de Dados e Aprendizado de Máquina](https://machinelearning.ghdaru.com.br/ii-5-arvores-ensembles.html).

Aqui o terreno muda: a fronteira entre as classes é **irregular**, e é onde a árvore ganha da reta. Você vai treinar os quatro modelos no mesmo dado e ver a diferença em número.

Só biblioteca padrão — o que significa que o treino é honesto e **lento**. As células levam alguns segundos cada; é o preço de não haver mágica compilada no meio.

In [ ]:
# --- roda igual na sua máquina e no Colab ------------------------------
# Na sua máquina: o notebook acha o repositório subindo de pasta.
# No Colab: não há repositório, então os arquivos necessários são baixados.
import pathlib, sys, urllib.request

RAW = "https://raw.githubusercontent.com/GHDaru/machinelearning/main/"
PRECISA = ['ml-zero/etapa-07/arvores.py', 'ml-zero/etapa-07/dados_tabulares.py', 'ml-zero/etapa-07/linear.py']

raiz = pathlib.Path.cwd()
for _ in range(5):
    if (raiz / "ml-zero").is_dir():
        break
    raiz = raiz.parent
else:
    raiz = pathlib.Path.cwd()

for rel in PRECISA:
    destino = raiz / rel
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(RAW + rel, destino)
        print("baixado:", rel)

sys.path.insert(0, str(raiz / "ml-zero/etapa-07"))
RAIZ = raiz
print("pronto.")

## 1. O terreno irregular

`n=1200` para o notebook rodar rápido. O experimento do capítulo usa 3000 e cinco seeds — está em `rodar.py`, e é de lá que saem os números publicados.

In [ ]:
from dados_tabulares import gerar, dividir

dados = gerar(n=1200)
treino, teste = dividir(dados)
print(f"treino {len(treino.y)} · teste {len(teste.y)} · "
      f"prevalência {sum(dados.y)/len(dados.y):.3f} · {len(dados.X[0])} atributos")

## 2. Quatro modelos, o mesmo dado

A métrica é **AUC**: independe do limiar, e é comparável entre modelos com escalas de saída diferentes.

In [ ]:
from arvores import Arvore, Floresta, Boosting, auc
from linear import LogisticaSimples

modelos = {
    "linear (logística)": LogisticaSimples(epocas=300),
    "árvore única":       Arvore(max_profundidade=6, min_folha=10),
    "floresta (bagging)": Floresta(n_arvores=20, max_profundidade=8, seed=0),
    "boosting":           Boosting(n_arvores=40, taxa=0.1, max_profundidade=3, seed=0),
}

resultados = {}
for nome, m in modelos.items():
    m.fit(treino.X, treino.y)
    escores = m.predict_proba(teste.X)
    resultados[nome] = auc(teste.y, escores)
    print(f"{nome:20s} AUC {resultados[nome]:.4f}")

## 3. Leia o resultado com cuidado

O linear perde **aqui** — e o capítulo II.2 existe para lembrar que isso é uma afirmação sobre **este terreno**, não sobre o modelo. O dado foi construído com uma fronteira irregular, que é exatamente onde a reta não tem chance.

A vantagem do ensemble é uma propriedade **do terreno**, não uma constante. Aumente o ruído e veja a distância entre os dois encolher — o sinal que a árvore explorava deixa de estar lá:

In [ ]:
for ruido in (0.0, 0.25):
    d = gerar(n=800, ruido=ruido)
    tr, te = dividir(d)
    linha = []
    for nome, construtor in [("linear", lambda: LogisticaSimples(epocas=300)),
                             ("boosting", lambda: Boosting(n_arvores=30, taxa=0.1, seed=0))]:
        m = construtor().fit(tr.X, tr.y)
        linha.append(f"{nome} {auc(te.y, m.predict_proba(te.X)):.4f}")
    print(f"ruído {ruido:.2f} · " + " · ".join(linha))

## O que levar

- O modelo linear perdeu **neste terreno**, não em geral. "Qual modelo é melhor" é pergunta mal formulada sem o dado na frente.
- **Floresta ataca variância** (média de modelos instáveis); **boosting ataca viés** (cada árvore corrige o resíduo da anterior). São remédios para doenças diferentes — a decomposição do capítulo 0.2.
- Ensemble costuma ganhar em tabular, e **paga** em interpretabilidade e calibração (capítulos II.1 e 14).

Continue em [07 — Árvores e Ensembles](https://machinelearning.ghdaru.com.br/ii-5-arvores-ensembles.html).